# Phase 4A1 — Stride-2 RG Correction Validation
## ThermoRG v8: Validate β = β₀ · φ^n_s with φ = 0.87

**Objective**: Test stride-2 multiplicative suppression across multiple architectures.

**Theory**:
- Each stride-2 layer suppresses β by factor φ ≈ 0.87
- Δ_stride = -log₂(φ) ≈ 0.20 (scaling dimension)
- Full model: β = β₀(J) · φ(γ) · φ^n_s

**Architectures**:
- ThermoNet (baseline, n_s=0): no stride-2
- ResNet-18 (n_s=3): conv1 + 3 downsample stages
- ResNet-34 (n_s=3): deeper version
- WideResNet-28-10 (n_s=3): wide + deep

**Success criterion**: RMSE(β_pred, β_obs) < 0.05 across architectures

## Step 1: Clone + Install

In [ ]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url} --branch develop
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
print("Cloned develop branch")

## Step 2: Imports + Setup

In [ ]:
import os, sys, json, math, time
import numpy as np
import torch
import torch.nn as nn
from torch import linalg
from pathlib import Path

sys.path.insert(0, '/kaggle/working/ThermoRG-NN')
sys.path.insert(0, '/kaggle/working/ThermoRG-NN/experiments/lift_test')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## Step 3: J_topo Computation (with Stride Correction)

In [ ]:
def compute_D_eff(W):
    """D_eff = ||W||_F² / ||W||_2²"""
    if W.dim() == 4:
        W = W.view(W.size(0), -1)
    fro_sq = (W ** 2).sum().item()
    S = linalg.svd(W.cpu()).S
    spec_sq = S[0].item() ** 2 + 1e-12
    return fro_sq / spec_sq

def compute_J_topo_stride(model, d_input=3.0):
    """
    J_topo with stride-2 RG correction.
    
    ζ_l = C_out / (C_in · s²)  for stride s > 1
    η_stride = η · ζ_l
    """
    skip_exclude = ['layernorm', 'layer_norm', 'norm', 'batchnorm', 
                    'bn', 'pool', 'flatten', 'fc', 'linear']
    
    eta_list = []
    prev_D_eff = None
    prev_c_out = None
    
    for name, module in model.named_modules():
        # Skip excluded layers
        if any(p in name.lower() for p in skip_exclude):
            prev_D_eff = None
            prev_c_out = None
            continue
        
        # Get weight
        if not isinstance(module, (nn.Conv2d, nn.Linear)):
            continue
        W = module.weight.data
        
        # Stride and channels
        s = 1
        c_in = W.shape[1]
        c_out = W.shape[0]
        if isinstance(module, nn.Conv2d):
            s = module.stride[0] if hasattr(module, 'stride') else 1
            c_in = module.in_channels
            c_out = module.out_channels
        
        D_eff = compute_D_eff(W)
        
        if prev_D_eff is not None and prev_c_out is not None:
            eta = D_eff / max(prev_D_eff, 1.0)
            # Stride correction: spatial-channel compression
            if s > 1:
                zeta = (c_out / prev_c_out) / (s ** 2)
                eta = eta * zeta
            eta_list.append(float(eta))
        
        prev_D_eff = D_eff
        prev_c_out = c_out
    
    if not eta_list:
        return 1.0, []
    
    log_etas = [abs(math.log(max(e, 1e-10))) for e in eta_list]
    J_topo = math.exp(-np.mean(log_etas))
    return float(J_topo), eta_list

def count_stride2_layers(model):
    """Count number of stride-2 convolutional layers."""
    n_stride2 = 0
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            if module.stride[0] == 2:
                n_stride2 += 1
    return n_stride2

## Step 4: Load Architectures

In [ ]:
from experiments.lift_test import ResNet18CIFAR

ARCHITECTURES = {
    'ThermoNet-L3': None,   # Will create below
    'ResNet-18': ResNet18CIFAR(),
}

# ThermoNet-L3 baseline (no stride-2)
class ThermoNetL3(nn.Module):
    def __init__(self, base_ch=64):
        super().__init__()
        self.blocks = nn.ModuleList([
            nn.Conv2d(3, base_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_ch),
            nn.GELU(),
            nn.Conv2d(base_ch, base_ch*2, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_ch*2),
            nn.GELU(),
            nn.Conv2d(base_ch*2, base_ch*2, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_ch*2),
            nn.GELU(),
        ])
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(base_ch*2, 10)
    
    def forward(self, x):
        for b in self.blocks:
            x = b(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)
    
    def get_conv_weights(self):
        return [m.weight.data for m in self.modules() 
                if isinstance(m, nn.Conv2d)]

ARCHITECTURES['ThermoNet-L3'] = ThermoNetL3(base_ch=64)

# Print architecture info
for name, arch in ARCHITECTURES.items():
    arch.eval()
    J, etas = compute_J_topo_stride(arch)
    n_s = count_stride2_layers(arch)
    n_params = sum(p.numel() for p in arch.parameters())
    print(f"{name:20s}: J_topo={J:.4f}, n_stride2={n_s}, params={n_params/1e6:.2f}M")

## Step 5: Compute J_topo and Count Stride-2 Layers

In [ ]:
# Compute J_topo and n_s for all architectures
results = []
for name, arch in ARCHITECTURES.items():
    arch.eval()
    J, etas = compute_J_topo_stride(arch)
    n_s = count_stride2_layers(arch)
    results.append({
        'name': name,
        'J_topo': J,
        'n_stride2': n_s,
        'n_eta_layers': len(etas)
    })
    print(f"{name}: J_topo={J:.4f}, n_s={n_s}, layers={len(etas)}")

print("\n--- Theoretical β predictions ---")
print("Using: β = β₀ · 0.87^n_s")
print("ThermoNet-L3: β₀ ≈ 0.86 (calibrated from Phase A)")
print()
for r in results:
    if r['name'] == 'ThermoNet-L3':
        beta_pred = 0.86
    else:
        beta_pred = 0.86 * (0.87 ** r['n_stride2'])
    r['beta_pred'] = beta_pred
    print(f"{r['name']:20s}: β_pred = {beta_pred:.4f} (n_s={r['n_stride2']})")

## Step 6: Training Setup (D-scaling)

In [ ]:
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Subset

D_VALUES = [2000, 5000, 10000, 25000]
EPOCHS = 100
BATCH_SIZE = 128
LR = 0.01
WD = 5e-4

# Transforms
transform_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])
transform_val = T.Compose([
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])

train_ds = CIFAR10(root='./data', train=True, transform=transform_train, download=True)
val_ds = CIFAR10(root='./data', train=False, transform=transform_val, download=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

def get_subset_loader(D, batch_size=BATCH_SIZE):
    """Get a dataloader with exactly D training samples."""
    indices = list(range(D))
    subset = Subset(train_ds, indices)
    return DataLoader(subset, batch_size=batch_size, shuffle=True, num_workers=2)

print(f"D values: {D_VALUES}")
print(f"Epochs: {EPOCHS}")

## Step 7: Training Function

In [ ]:
def train_and_evaluate(model, train_loader, val_loader, epochs, lr, wd):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    best_epoch = 0
    
    for epoch in range(epochs):
        model.train()
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
        scheduler.step()
        
        # Evaluate
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                out = model(X)
                val_loss += criterion(out, y).item() * X.size(0)
                correct += (out.argmax(1) == y).sum().item()
                total += X.size(0)
        val_loss /= total
        val_acc = correct / total
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_epoch = epoch + 1
    
    return {
        'best_val_loss': best_val_loss,
        'best_val_acc': best_val_acc,
        'best_epoch': best_epoch
    }

## Step 8: Run D-Scaling Experiment

In [ ]:
from scipy.optimize import curve_fit

def fit_scaling_law(losses_by_d, d_values):
    """Fit L(D) = α·D^(-β) + E"""
    def power_law(D, alpha, beta, E):
        return alpha * np.array(D) ** (-beta) + E
    
    Ds = np.array(d_values)
    losses = np.array([losses_by_d[d] for d in d_values])
    
    try:
        popt, pcov = curve_fit(power_law, Ds, losses,
                               p0=[10.0, 0.5, 0.5],
                               bounds=([0, 0, 0], [1000, 5, 10]),
                               maxfev=10000)
        alpha, beta, E = popt
        preds = power_law(Ds, *popt)
        ss_res = ((losses - preds) ** 2).sum()
        ss_tot = ((losses - losses.mean()) ** 2).sum()
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        return {'alpha': float(alpha), 'beta': float(beta), 
                'E': float(E), 'R2': float(r2)}
    except Exception as e:
        return {'alpha': None, 'beta': None, 'E': None, 
                'R2': 0.0, 'error': str(e)}

print("Starting D-scaling experiment...")
all_results = {}

for arch_name, arch_template in ARCHITECTURES.items():
    print(f"\n{'='*60}")
    print(f"Architecture: {arch_name}")
    print(f"{'='*60}")
    
    arch_results = {'D_results': {}, 'J_topo': None, 'n_stride2': None}
    
    # Get J_topo and n_stride2
    arch_template.eval()
    J, _ = compute_J_topo_stride(arch_template)
    n_s = count_stride2_layers(arch_template)
    arch_results['J_topo'] = J
    arch_results['n_stride2'] = n_s
    print(f"J_topo = {J:.4f}, n_stride2 = {n_s}")
    
    losses_by_d = {}
    
    for D in D_VALUES:
        print(f"  D={D}: ", end='', flush=True)
        loader = get_subset_loader(D)
        
        # Fresh model for each D
        if arch_name == 'ResNet-18':
            model = ResNet18CIFAR().to(DEVICE)
        else:
            model = ThermoNetL3(base_ch=64).to(DEVICE)
        
        result = train_and_evaluate(model, loader, val_loader, 
                                    EPOCHS, LR, WD)
        losses_by_d[D] = result['best_val_loss']
        arch_results['D_results'][str(D)] = result
        print(f"loss={result['best_val_loss']:.4f}")
        
        del model
        torch.cuda.empty_cache()
    
    # Fit scaling law
    fit = fit_scaling_law(losses_by_d, D_VALUES)
    arch_results['scaling_fit'] = fit
    print(f"  Scaling fit: α={fit.get('alpha', 'N/A'):.2f}, "
          f"β={fit.get('beta', 'N/A'):.4f}, E={fit.get('E', 'N/A'):.4f}, "
          f"R²={fit.get('R2', 'N/A'):.4f}")
    
    all_results[arch_name] = arch_results

print("\nDone!")

## Step 9: Analysis — Compare β Predictions vs Observations

In [ ]:
print("\n" + "="*70)
print("RESULTS: Stride-2 β Suppression Validation")
print("="*70)

print(f"\n{'Architecture':<20} {'n_s':>4} {'J_topo':>8} {'β_pred':>8} {'β_obs':>8} {'Error':>8}")
print("-"*70)

errors = []
for arch_name, res in all_results.items():
    n_s = res['n_stride2']
    J = res['J_topo']
    beta_pred = res.get('beta_pred', 0.86 * (0.87 ** n_s))
    beta_obs = res['scaling_fit'].get('beta', None)
    
    if beta_obs is not None:
        err = beta_obs - beta_pred
        errors.append(err)
        print(f"{arch_name:<20} {n_s:>4} {J:>8.4f} {beta_pred:>8.4f} {beta_obs:>8.4f} {err:>8.4f}")
    else:
        print(f"{arch_name:<20} {n_s:>4} {J:>8.4f} {beta_pred:>8.4f} {'N/A':>8} {'N/A':>8}")

if errors:
    rmse = math.sqrt(np.mean(np.array(errors)**2))
    mae = np.mean(np.abs(errors))
    print(f"\nRMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"Success criterion (RMSE < 0.05): {'✅ PASS' if rmse < 0.05 else '❌ FAIL'}")

## Step 10: Save Results

In [ ]:
out_dir = Path('./phase_4a1_results')
out_dir.mkdir(exist_ok=True)

# Convert non-serializable items
def make_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (np.floating, np.integer)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj

serializable_results = make_serializable(all_results)

with open(out_dir / 'stride2_validation_results.json', 'w') as f:
    json.dump(serializable_results, f, indent=2)

print(f"Results saved to {out_dir / 'stride2_validation_results.json'}")

## Step 11: Git Push

In [ ]:
%cd /kaggle/working/ThermoRG-NN
!git add -A && git commit -m "Phase 4A1: Stride-2 RG correction validation results"
!git push origin develop